# Sandbox: cohort variants on the cross-sectional analyses

Reads the manuscript inputs and writes only to `sandbox/`. The last cell confirms the manuscript files are unchanged.

| cohort | definition |
|---|---|
| A0_record | cohort of record (36 / 12 / 12) |
| A1_no010 | A0 without sub-010 |
| A2_no010_hemi | A1, Hemi patients only (resection type from `marlene_lmm.py` SURG) |
| A3_prisma | A0, keeping subjects whose selected session was on the Prisma |

Removals filter the existing ComBat-harmonized values, as the current exclusions do; nothing is refit.
Stats come from `D_liu/verified/05_stats_harmony.py`, unchanged, once per cohort (a few minutes each, cached in `sandbox/<cohort>/`).
Not in this pass: bambi evidence ratios, Tables 2 and 4, TFCE, figures.

Kernel: fmri. Run top to bottom.


## 1 · Config

In [1]:
# ─── Config ──────────────────────────────────────────────────────────────────
# Reads the manuscript inputs and writes only to sandbox/. The manuscript files
# are hashed here and again in the last cell, so any write to them is caught.
import sys, json, time, hashlib, contextlib, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

sys.dont_write_bytecode = True   # loading the stats script must not write __pycache__ into the repo

T_START = time.time()
REPO    = Path('/user_data/csimmon2/git_repos/sym_pt')
SANDBOX = REPO / 'sandbox'
SANDBOX.mkdir(exist_ok=True)

STATS_SCRIPT = REPO / 'D_liu' / 'verified' / '05_stats_harmony.py'
UNIVAR_SQRT  = REPO / 'D_liu' / 'univariate_v1_harmonized_sqrt.csv'
RSA_HARM     = REPO / 'D_liu' / 'rsa_v1_harmonized.csv'
SCANNER_CSV  = REPO / 'F_harmonization' / 'sub_info_scanner.csv'
RSM_CSV      = REPO / 'H_RosenkeRSM' / 'otc_rsm_persubject.csv'

# manuscript stats: read for the A0 check only
STATS_RECORD = REPO / 'D_liu' / 'stats_results_harmonized_corrected.csv'
STATS_PTPT4  = REPO / 'D_liu' / 'stats_results_harmonized_corrected_ptpt4.csv'  # compared if present
GUARDED = [STATS_RECORD, STATS_PTPT4, REPO / 'D_liu' / 'tfce_clusters.csv',
           UNIVAR_SQRT, RSA_HARM, RSM_CSV, SCANNER_CSV, STATS_SCRIPT]

# cohort of record: EXCLUDE / EXCLUDE_SES as in 05_stats_harmony.py and final_figures cell 2
EXCLUDE_RECORD = ['sub-017', 'sub-091', 'sub-095', 'sub-096', 'sub-027', 'sub-084']
EXCLUDE_SES    = [('sub-108', 2)]

# resection type, copied from marlene_lmm.py (SURG, line 71 on). sub-091 falls to the age cap.
SURG = {
    'sub-077': 'Hemi', 'sub-004': 'OTC',  'sub-075': 'OTC',  'sub-074': 'Hemi',
    'sub-008': 'Hemi', 'sub-098': 'Hemi', 'sub-079': 'OTC',  'sub-039': 'OTC',
    'sub-109': 'Hemi', 'sub-089': 'OTC',  'sub-101': 'Hemi', 'sub-005': 'OTC',
    'sub-091': 'Hemi', 'sub-099': 'Hemi', 'sub-069': 'Hemi', 'sub-044': 'OTC',
    'sub-010': 'Hemi', 'sub-021': 'OTC',  'sub-090': 'Hemi', 'sub-076': 'OTC',
    'sub-108': 'Hemi', 'sub-066': 'Hemi', 'sub-092': 'Hemi', 'sub-082': 'OTC',
    'sub-078': 'OTC',
}

FORCE = False   # True reruns the stats even when a cached run with identical inputs exists


def md5(p):
    p = Path(p)
    return hashlib.md5(p.read_bytes()).hexdigest() if p.exists() else None


HASH_BEFORE = {str(p): md5(p) for p in GUARDED}
for p in [STATS_SCRIPT, UNIVAR_SQRT, RSA_HARM, SCANNER_CSV, RSM_CSV, STATS_RECORD]:
    print(('ok       ' if p.exists() else 'MISSING  ') + str(p))

ok       /user_data/csimmon2/git_repos/sym_pt/D_liu/verified/05_stats_harmony.py
ok       /user_data/csimmon2/git_repos/sym_pt/D_liu/univariate_v1_harmonized_sqrt.csv
ok       /user_data/csimmon2/git_repos/sym_pt/D_liu/rsa_v1_harmonized.csv
ok       /user_data/csimmon2/git_repos/sym_pt/F_harmonization/sub_info_scanner.csv
ok       /user_data/csimmon2/git_repos/sym_pt/H_RosenkeRSM/otc_rsm_persubject.csv
ok       /user_data/csimmon2/git_repos/sym_pt/D_liu/stats_results_harmonized_corrected.csv


## 2 · Cohorts

In [2]:
# ─── Cohorts ─────────────────────────────────────────────────────────────────
# Same session rule as 05_stats_harmony.py: controls first session, OTC patients
# last session, after EXCLUDE and EXCLUDE_SES. Checked in both harmonized files.
# A3 keeps a subject only if its selected session was acquired on the Prisma.

def _ses_num(s):
    return pd.to_numeric(s.astype(str).str.extract(r'(\d+)', expand=False),
                         errors='coerce')


def selected_sessions(path, exclude):
    d = pd.read_csv(path, usecols=lambda c: c in
                    {'subject_id', 'session', 'group', 'status', 'intact_hemi'})
    d = d[~d['subject_id'].isin(exclude)].copy()
    d['ses_num'] = _ses_num(d['session'])
    for s, n in EXCLUDE_SES:
        d = d[~((d['subject_id'] == s) & (d['ses_num'] == n))]
    c = d[d['status'] == 'control']
    c = c[c['ses_num'] == c.groupby('subject_id')['ses_num'].transform('min')]
    p = d[d['group'] == 'OTC']
    p = p[p['ses_num'] == p.groupby('subject_id')['ses_num'].transform('max')]
    c = c.assign(cell='control')
    p = p.assign(cell=p['intact_hemi'].map({'left': 'LH-intact', 'right': 'RH-intact'}))
    return pd.concat([c, p])[['subject_id', 'ses_num', 'cell']].drop_duplicates()


sel_u = selected_sessions(UNIVAR_SQRT, EXCLUDE_RECORD)
sel_r = selected_sessions(RSA_HARM, EXCLUDE_RECORD)
_cmp = sel_u.merge(sel_r, on=['subject_id', 'cell'], how='outer',
                   suffixes=('_univariate', '_rsa'), indicator=True)
_odd = _cmp[(_cmp['_merge'] != 'both') |
            (_cmp['ses_num_univariate'] != _cmp['ses_num_rsa'])]
if len(_odd):
    print('selected session differs between the two harmonized files:')
    print(_odd.drop(columns='_merge').to_string(index=False), '\n')

BASE = pd.concat([sel_u, sel_r]).drop_duplicates(['subject_id', 'ses_num', 'cell'])
if BASE['cell'].isna().any():
    raise ValueError('patients with no intact_hemi:\n'
                     + BASE[BASE['cell'].isna()].to_string(index=False))

_sc = pd.read_csv(SCANNER_CSV)
_sc = pd.DataFrame({'subject_id': _sc['sub'], 'ses_num': _ses_num(_sc['ses']),
                    'scanner': _sc['scanner']}).drop_duplicates(['subject_id', 'ses_num'])
BASE = BASE.merge(_sc, on=['subject_id', 'ses_num'], how='left')
BASE['surg'] = BASE['subject_id'].map(SURG)

_no_sc = BASE[BASE['scanner'].isna()]
_no_sg = BASE[(BASE['cell'] != 'control') & BASE['surg'].isna()]
if len(_no_sc) or len(_no_sg):
    raise ValueError('missing labels:\n'
                     + (f'no scanner row:\n{_no_sc.to_string(index=False)}\n' if len(_no_sc) else '')
                     + (f'no SURG entry:\n{_no_sg.to_string(index=False)}' if len(_no_sg) else ''))

_pts = BASE[BASE['cell'] != 'control']
COHORTS = {
    'A0_record':     [],
    'A1_no010':      ['sub-010'],
    'A2_no010_hemi': sorted({'sub-010'} | set(_pts.loc[_pts['surg'] != 'Hemi', 'subject_id'])),
    'A3_prisma':     sorted(BASE.loc[BASE['scanner'] != 'Prisma', 'subject_id'].unique()),
}
EXCLUDES = {k: EXCLUDE_RECORD + v for k, v in COHORTS.items()}
if 'sub-010' not in set(BASE['subject_id']):
    print('NOTE: sub-010 is not in the cohort of record, so A1 equals A0')

_rows = []
for k, extra in COHORTS.items():
    n = BASE[~BASE['subject_id'].isin(extra)].drop_duplicates('subject_id')['cell'].value_counts()
    _rows.append({'cohort': k, 'controls': n.get('control', 0),
                  'LH-intact': n.get('LH-intact', 0), 'RH-intact': n.get('RH-intact', 0),
                  'dropped beyond the record': len(extra)})
COHORT_N = pd.DataFrame(_rows).set_index('cohort')
print(COHORT_N.to_string())
_n0 = COHORT_N.loc['A0_record']
if (_n0['controls'], _n0['LH-intact'], _n0['RH-intact']) != (36, 12, 12):
    print('\nWARNING: A0 is not 36 / 12 / 12')
for k, extra in COHORTS.items():
    if extra:
        print(f'\n{k} drops: ' + ', '.join(extra))

               controls  LH-intact  RH-intact  dropped beyond the record
cohort                                                                  
A0_record            36         12         12                          0
A1_no010             36         12         11                          1
A2_no010_hemi        36          6          6                         12
A3_prisma            13          8          9                         30

A1_no010 drops: sub-010

A2_no010_hemi drops: sub-004, sub-005, sub-010, sub-021, sub-039, sub-044, sub-075, sub-076, sub-078, sub-079, sub-082, sub-089

A3_prisma drops: sub-001, sub-004, sub-005, sub-008, sub-010, sub-016, sub-018, sub-021, sub-022, sub-023, sub-025, sub-026, sub-028, sub-029, sub-031, sub-034, sub-035, sub-036, sub-037, sub-039, sub-040, sub-041, sub-043, sub-044, sub-052, sub-054, sub-058, sub-062, sub-064, sub-065


## 3 · Stats per cohort

In [3]:
# ─── Stats: 05_stats_harmony.py, unchanged, once per cohort ──────────────────
# The script is loaded fresh for each cohort, so its RNG restarts at seed 42
# and A0 should reproduce the manuscript stats file exactly. Only its exclusion
# lists and output paths are replaced. Output and log go to sandbox/<cohort>/;
# a cached run is reused when the exclusions, script and inputs are unchanged.

def run_stats(name, exclude):
    out = SANDBOX / name
    out.mkdir(exist_ok=True)
    res, meta_f, log_f = out / 'stats_results.csv', out / 'meta.json', out / 'stats_log.txt'
    meta = json.loads(json.dumps({
        'exclude': sorted(exclude), 'exclude_ses': EXCLUDE_SES,
        'script_md5': md5(STATS_SCRIPT), 'univar_md5': md5(UNIVAR_SQRT),
        'rsa_md5': md5(RSA_HARM)}))
    if (not FORCE and res.exists() and meta_f.exists()
            and json.loads(meta_f.read_text()) == meta):
        print(f'{name}: cached ({res})')
        return pd.read_csv(res)

    spec = importlib.util.spec_from_file_location(f'stats05_{name}', STATS_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)                     # fresh module, fresh RNG
    mod.EXCLUDE, mod.EXCLUDE_SES = list(exclude), list(EXCLUDE_SES)
    mod.OUT_RESULTS, mod.OUT_TFCE = res, out / 'tfce_clusters.csv'

    argv, t0 = sys.argv, time.time()
    sys.argv = [str(STATS_SCRIPT)]                   # the kernel's own argv would break argparse
    try:
        with open(log_f, 'w') as log, contextlib.redirect_stdout(log), \
                contextlib.redirect_stderr(log):
            mod.main()
    except Exception:
        print(f'{name}: FAILED - last lines of {log_f}:')
        print(''.join(open(log_f).readlines()[-15:]))
        raise
    finally:
        sys.argv = argv
    meta_f.write_text(json.dumps(meta, indent=1))
    print(f'{name}: {time.time() - t0:.0f} s')
    return pd.read_csv(res)


STATS = {k: run_stats(k, v) for k, v in EXCLUDES.items()}

A0_record: 117 s
A1_no010: 111 s
A2_no010_hemi: 111 s
A3_prisma: 106 s


## 4 · Check: A0 against the manuscript stats file

In [4]:
# ─── Check: does A0 reproduce the manuscript stats file? ─────────────────────
# Compares every numeric column. Row by row when both files list the same tests
# in the same order, otherwise matched on measure/model/comparison/hemi/level.

KEYS = ['measure', 'model', 'comparison', 'hemi', 'level']


def frame_diff(new, old, tol=1e-9):
    a, b = new.copy(), old.copy()
    for d in (a, b):
        d[KEYS] = d[KEYS].fillna('').astype(str)
    if len(a) == len(b) and (a[KEYS].values == b[KEYS].values).all():
        pairs, how = (a.reset_index(drop=True), b.reset_index(drop=True)), 'row by row'
    else:
        dup = int(a.duplicated(KEYS).sum() + b.duplicated(KEYS).sum())
        m = a.merge(b, on=KEYS, how='outer', suffixes=('', '__old'), indicator=True)
        only = m[m['_merge'] != 'both']
        how = (f'by key ({len(a)} vs {len(b)} rows, {len(only)} unmatched, '
               f'{dup} duplicate keys)')
        m = m[m['_merge'] == 'both']
        pairs = (m[[c for c in m.columns if not c.endswith('__old')]],
                 m[KEYS + [c for c in m.columns if c.endswith('__old')]]
                 .rename(columns=lambda c: c.replace('__old', '')))
    x, y = pairs
    num = [c for c in y.columns if c in x.columns and c not in KEYS
           and pd.api.types.is_numeric_dtype(y[c])]
    bad = []
    for c in num:
        u = pd.to_numeric(x[c], errors='coerce').to_numpy(float)
        v = pd.to_numeric(y[c], errors='coerce').to_numpy(float)
        dd = np.abs(u - v)
        dd[np.isnan(u) & np.isnan(v)] = 0
        dd[np.isnan(dd)] = np.inf
        for i in np.where(dd > tol)[0]:
            bad.append({**{k: x[k].iloc[i] for k in KEYS}, 'column': c,
                        'record': v[i], 'sandbox': u[i]})
    return how, pd.DataFrame(bad)


for ref in [STATS_RECORD, STATS_PTPT4]:
    if not ref.exists():
        continue
    how, bad = frame_diff(STATS['A0_record'], pd.read_csv(ref))
    print(f'A0 vs {ref.name}: compared {how}')
    if bad.empty:
        print('  PASS: every value identical\n')
    else:
        print(f'  DIFFERS: {len(bad)} values in {bad[KEYS].drop_duplicates().shape[0]} rows; '
              f'max |diff| {np.nanmax(np.abs(bad.sandbox - bad.record)):.3g}')
        print(bad.head(15).to_string(index=False), '\n')

A0 vs stats_results_harmonized_corrected.csv: compared by key (274 vs 242 rows, 32 unmatched, 0 duplicate keys)
  DIFFERS: 243 values in 63 rows; max |diff| 0.0607
 measure                                 model         comparison hemi           level column   record  sandbox
geometry house_PPA_strict_LH-intact_pt_vs_ctrl  group_main_effect      MEAN_OVER_PAIRS p_perm 0.712829 0.711529
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patient_vs_control           face-house p_perm 0.953305 0.959304
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patient_vs_control          face-object p_perm 0.357664 0.360664
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patient_vs_control            face-word p_perm 0.161384 0.168383
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patient_vs_control         house-object p_perm 0.994101 0.994201
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patient_vs_control           house-word p_perm 0.246775 0.243376
geometry house_PPA_strict_LH-intact_pt_vs_ctrl patie

## 5 · Whole-OTC per cohort

In [5]:
# ─── Whole-OTC: side typicality, consistency, arrangement ────────────────────
# Functions ported from whole_otc_bayes.ipynb: _zc/_corr and side typicality
# (cell 3), profile-shape permutation and consistency (cell 19), JZS BF (cell 8).
# otc_rsm_persubject.csv already holds the cohort of record, so each cohort is a
# filter of it. Fresh RNGs per cohort (seeds as in the source cells).

from scipy import integrate

_ch_all = pd.read_csv(RSM_CSV)
P6 = [c for c in _ch_all.columns if '-' in c]
assert len(P6) == 6, P6
_ch_all['set'] = np.where(
    _ch_all.group == 'control', 'ctl_' + _ch_all.hemi.str.upper(),
    np.where(_ch_all.intact_hemi == _ch_all.hemi.map({'l': 'left', 'r': 'right'}),
             'pt_' + _ch_all.hemi.str.upper(), 'drop'))
_ch_all = _ch_all[_ch_all['set'] != 'drop'].reset_index(drop=True)
SETS = ['ctl_L', 'ctl_R', 'pt_L', 'pt_R']


def _zc(M):
    return (M - M.mean(1, keepdims=True)) / M.std(1, ddof=0, keepdims=True)


def _corr(M):
    Z = _zc(M)
    return Z @ Z.T / M.shape[1]


def _shape_stat(d):
    return float(((d - d.mean()) ** 2).sum())


def bf10_jzs(x, r=0.707):
    """One-sample JZS BF10 against 0 (Rouder et al. 2009), as whole_otc_bayes cell 8."""
    x = np.asarray(x, float)
    t, n = x.mean() / (x.std(ddof=1) / np.sqrt(len(x))), len(x)
    v = n - 1

    def num(g):
        return ((1 + n * g) ** -0.5 * (1 + t ** 2 / ((1 + n * g) * v)) ** (-(v + 1) / 2)
                * (r ** 2 / 2) ** 0.5 / np.sqrt(np.pi) * g ** -1.5 * np.exp(-r ** 2 / (2 * g)))
    return integrate.quad(num, 0, np.inf)[0] / (1 + t ** 2 / v) ** (-(v + 1) / 2)


def whole_otc(exclude, nperm=10000):
    ch = _ch_all[~_ch_all['subject_id'].isin(exclude)].reset_index(drop=True)
    M = ch[P6].to_numpy(float)
    sid, st = ch['subject_id'].to_numpy(), ch['set'].to_numpy()
    SAME = sid[:, None] == sid[None, :]
    IDX = {s: np.where(st == s)[0] for s in SETS}
    C = _corr(M)

    def blk(i, j):
        b, keep = C[np.ix_(i, j)], ~SAME[np.ix_(i, j)]
        return b[keep].mean()

    typ = np.array([blk([i], IDX['ctl_' + st[i][-1]])
                    - blk([i], IDX['ctl_' + ('R' if st[i].endswith('L') else 'L')])
                    for i in range(len(M))])

    def consistency(idx):
        Z = _zc(M[idx]); Cc = Z @ Z.T / M.shape[1]
        np.fill_diagonal(Cc, np.nan)
        return np.nanmean(Cc, axis=1)

    rng_shape, rng_perm = np.random.default_rng(41), np.random.default_rng(42)

    def shape(sa, sb, paired):
        A, B = M[IDX[sa]], M[IDX[sb]]
        if paired:
            A, B = A[np.argsort(sid[IDX[sa]])], B[np.argsort(sid[IDX[sb]])]
            D = A - B
            obs = _shape_stat(D.mean(0))
            null = np.array([_shape_stat((D * rng_shape.choice([-1, 1], len(D))[:, None]).mean(0))
                             for _ in range(nperm)])
        else:
            obs = _shape_stat(A.mean(0) - B.mean(0))
            pool, na = np.vstack([A, B]), len(A)
            null = np.empty(nperm)
            for b in range(nperm):
                p = rng_shape.permutation(len(pool))
                null[b] = _shape_stat(pool[p[:na]].mean(0) - pool[p[na:]].mean(0))
        return (np.sum(null >= obs) + 1) / (nperm + 1)

    def perm1(a, b):
        obs = a.mean() - b.mean()
        pool, na, k = np.concatenate([a, b]), len(a), 0
        for _ in range(nperm):
            p = rng_perm.permutation(pool)
            k += abs(p[:na].mean() - p[na:].mean()) >= abs(obs) - 1e-12
        return (k + 1) / (nperm + 1)

    cons = {s: consistency(IDX[s]) for s in SETS}
    out = {f'n {s}': len(IDX[s]) for s in SETS}
    out.update({f'typicality {s}': typ[IDX[s]].mean() for s in SETS})
    out['typicality controls'] = typ[np.r_[IDX['ctl_L'], IDX['ctl_R']]].mean()
    out['BF10 typicality pt_L'] = bf10_jzs(typ[IDX['pt_L']])
    out['BF10 typicality pt_R'] = bf10_jzs(typ[IDX['pt_R']])
    out.update({f'consistency {s}': cons[s].mean() for s in SETS})
    out.update({f'consistency SD {s}': cons[s].std(ddof=1) for s in SETS})
    # same order as whole_otc_bayes cell 19 first, then the patient-vs-control pairs
    out['arrangement p ctl_L vs ctl_R'] = shape('ctl_L', 'ctl_R', paired=True)
    out['arrangement p pt_L vs pt_R'] = shape('pt_L', 'pt_R', paired=False)
    out['arrangement p pt_L vs ctl_L'] = shape('pt_L', 'ctl_L', paired=False)
    out['arrangement p pt_R vs ctl_R'] = shape('pt_R', 'ctl_R', paired=False)
    out['consistency p pt_L vs pt_R'] = perm1(cons['pt_L'], cons['pt_R'])
    out['consistency p pt_L vs ctl_L'] = perm1(cons['pt_L'], cons['ctl_L'])
    out['consistency p pt_R vs ctl_R'] = perm1(cons['pt_R'], cons['ctl_R'])
    return out


WOTC = pd.DataFrame({k: whole_otc(v) for k, v in EXCLUDES.items()})

# manuscript values (V3 Results) next to A0; deterministic values should match to rounding
V3_WOTC = {'typicality ctl_L': .147, 'typicality ctl_R': .050, 'typicality pt_L': .002,
           'typicality pt_R': -.008, 'typicality controls': .098,
           'BF10 typicality pt_L': .29, 'BF10 typicality pt_R': .29,
           'consistency ctl_L': .703, 'consistency ctl_R': .606,
           'consistency pt_L': .419, 'consistency pt_R': .533,
           'arrangement p ctl_L vs ctl_R': .0001, 'arrangement p pt_L vs pt_R': .969,
           'arrangement p pt_L vs ctl_L': .025, 'arrangement p pt_R vs ctl_R': .315,
           'consistency p pt_L vs pt_R': .105, 'consistency p pt_R vs ctl_R': .133}
WOTC.insert(0, 'V3', pd.Series(V3_WOTC))
_tol = {k: (.01 if ' p ' in k else .0006 if 'BF' not in k else .006) for k in V3_WOTC}
_off = [k for k in V3_WOTC if abs(WOTC.loc[k, 'A0_record'] - V3_WOTC[k]) > _tol[k]]
print('whole-OTC A0 vs V3:', 'all within rounding / permutation error' if not _off
      else 'OFF for ' + ', '.join(_off))
with pd.option_context('display.float_format', '{:.3f}'.format):
    display(WOTC)

whole-OTC A0 vs V3: all within rounding / permutation error


,V3,A0_record,A1_no010,A2_no010_hemi,A3_prisma
n ctl_L,NaN,36.000,36.000,36.000,13.000
n ctl_R,NaN,36.000,36.000,36.000,13.000
n pt_L,NaN,12.000,12.000,6.000,8.000
n pt_R,NaN,12.000,11.000,6.000,9.000
typicality ctl_L,0.147,0.147,0.147,0.147,0.077
typicality ctl_R,0.050,0.050,0.050,0.050,0.134
typicality pt_L,0.002,0.002,0.002,-0.001,-0.041
typicality pt_R,-0.008,-0.008,-0.021,-0.058,0.066
typicality controls,0.098,0.098,0.098,0.098,0.106
BF10 typicality pt_L,0.290,0.288,0.288,0.373,0.367


## 6 · Table 1 per cohort, and all reported tests

In [6]:
# ─── Table 1 per cohort, and every reported test side by side ────────────────
# Table 1 cells follow V3: regions surviving FDR for peak selectivity and voxel
# count; for between-category similarity, ROIs whose pair x group arrangement
# test is p < .05 (the rule that reproduces V3's Table 1 rows). Bayes factors
# other than whole-OTC typicality are not recomputed in this pass.

ROI_SHORT = {'word_VWFA': 'VWFA', 'face_FFA': 'FFA', 'house_PPA_strict': 'PPA',
             'object_LOC': 'LO'}
ROI_ORDER = list(ROI_SHORT)
COMP = [('ctrl L vs R',       'ctrl_LvsR',            'control_asymmetry'),
        ('LH-intact vs ctrl', 'LH-intact_pt_vs_ctrl', 'patient_vs_control'),
        ('RH-intact vs ctrl', 'RH-intact_pt_vs_ctrl', 'patient_vs_control'),
        ('pt vs pt',          'pt_LH_vs_RH',          'patient_subgroup')]
V3_TABLE1 = pd.DataFrame(
    [['PPA, LO', 'VWFA, FFA, PPA, LO', 'VWFA, FFA, PPA', '+0.098'],
     ['LO', 'LO', 'FFA', '+0.002'],
     ['—', 'PPA, LO', 'VWFA, FFA, PPA, LO', '-0.008'],
     ['—', '—', '—', '—']],
    index=[c[0] for c in COMP],
    columns=['peak selectivity', 'selective voxels', 'between-category', 'own side'])


def _sel(st, **kw):
    m = np.ones(len(st), bool)
    for k, v in kw.items():
        m &= st[k].astype(str).values == str(v)
    return st[m]


def table1(st, wo):
    rows = {}
    for lab, model, comp in COMP:
        cell = {}
        for meas, col in [('peak_selectivity', 'peak selectivity'),
                          ('selective_voxels', 'selective voxels')]:
            d = _sel(st, measure=meas, model=model, comparison=comp)
            cell[col] = ', '.join(ROI_SHORT[r] for r in ROI_ORDER
                                  if (d[d['level'] == r]['q_fdr'] < .05).any()) or '—'
        g = _sel(st, measure='geometry', comparison='omnibus')
        cell['between-category'] = ', '.join(
            ROI_SHORT[r] for r in ROI_ORDER
            if (g[g['model'] == f'{r}_{model}']['omnibus_p'] < .05).any()) or '—'
        cell['own side'] = {'ctrl L vs R': f"{wo['typicality controls']:+.3f}",
                            'LH-intact vs ctrl': f"{wo['typicality pt_L']:+.3f}",
                            'RH-intact vs ctrl': f"{wo['typicality pt_R']:+.3f}",
                            'pt vs pt': '—'}[lab]
        rows[lab] = cell
    return pd.DataFrame(rows).T[V3_TABLE1.columns]


TABLE1 = {k: table1(STATS[k], WOTC[k]) for k in STATS}
pd.concat(TABLE1, names=['cohort']).to_csv(SANDBOX / 'table1_by_cohort.csv')

print('A0 vs V3 Table 1 (cells that differ):')
_a0 = TABLE1['A0_record']
_diff = [(r, c, V3_TABLE1.loc[r, c], _a0.loc[r, c]) for r in V3_TABLE1.index
         for c in V3_TABLE1.columns
         if V3_TABLE1.loc[r, c].replace('−', '-') != _a0.loc[r, c]]
print('  none' if not _diff else
      '\n'.join(f'  {r} | {c}: V3 "{v}"  sandbox "{s}"' for r, c, v, s in _diff))

for k, t in TABLE1.items():
    print(f'\n── {k}   (controls {COHORT_N.loc[k, "controls"]}, '
          f'LH-intact {COHORT_N.loc[k, "LH-intact"]}, RH-intact {COHORT_N.loc[k, "RH-intact"]})')
    print(t.to_string())
    if k != 'A0_record':
        ch = [(r, c, _a0.loc[r, c], t.loc[r, c]) for r in t.index for c in t.columns
              if c != 'own side' and t.loc[r, c] != _a0.loc[r, c]]
        print('  changed from A0: ' + ('none' if not ch else
              '; '.join(f'{r} | {c}: "{a}" -> "{b}"' for r, c, a, b in ch)))


# ── every test the Results report, one row each, cohorts side by side ──────
def _fmt(r, stat):
    if r is None:
        return ''
    if stat == 'omnibus':
        return f"p={r['omnibus_p']:.3f}" + (' *' if r['omnibus_p'] < .05 else '')
    p = r['q_fdr'] if stat == 'q' else r['p_perm']
    lab = 'q' if stat == 'q' else 'p'
    return f"d={r['cohens_d']:+.2f} {lab}={p:.3f}" + (' *' if p < .05 else '')


def _one(st, **kw):
    d = _sel(st, **kw)
    return None if d.empty else d.iloc[0]


spec = []
for meas in ['peak_selectivity', 'selective_voxels']:
    for lab, model, comp in COMP:
        if model != 'pt_LH_vs_RH':   # no pt-vs-pt omnibus in the stats file (it comes from marlene_lmm)
            spec.append((meas, lab, 'omnibus', dict(measure=meas, model=model, comparison='omnibus'), 'omnibus'))
        spec.append((meas, lab, 'mean over ROIs', dict(measure=meas, model=model,
                     comparison='group_main_effect', level='MEAN_OVER_ROIS'), 'p'))
        for roi in ROI_ORDER:
            spec.append((meas, lab, ROI_SHORT[roi], dict(measure=meas, model=model,
                         comparison=comp, level=roi), 'q'))
for roi in ROI_ORDER:
    for lab, model, comp in COMP:
        spec.append(('geometry', lab, f'{ROI_SHORT[roi]} arrangement',
                     dict(measure='geometry', model=f'{roi}_{model}', comparison='omnibus'), 'omnibus'))
        spec.append(('geometry', lab, f'{ROI_SHORT[roi]} mean of six',
                     dict(measure='geometry', model=f'{roi}_{model}',
                          comparison='group_main_effect', level='MEAN_OVER_PAIRS'), 'p'))
        for pair in ['face-house', 'face-object', 'face-word',
                     'house-object', 'house-word', 'object-word']:
            hit = [_one(STATS[k], measure='geometry', model=f'{roi}_{model}',
                        comparison=comp, level=pair) for k in STATS]
            if any(h is not None and h['q_fdr'] < .05 for h in hit):   # pairs significant in any cohort
                spec.append(('geometry', lab, f'{ROI_SHORT[roi]} {pair}',
                             dict(measure='geometry', model=f'{roi}_{model}',
                                  comparison=comp, level=pair), 'q'))
for lab, model, comp in COMP[1:3]:
    for roi in ROI_ORDER:
        spec.append(('peak_distance', lab, ROI_SHORT[roi], dict(measure='peak_distance',
                     model=model, comparison='patient_vs_control', level=roi), 'q'))

ALL = pd.DataFrame([{'measure': m, 'comparison': lab, 'test': t,
                     **{k: _fmt(_one(STATS[k], **kw), stat) for k in STATS}}
                    for m, lab, t, kw, stat in spec])
ALL.to_csv(SANDBOX / 'all_tests_by_cohort.csv', index=False)
print(f'\nAll reported tests by cohort ({len(ALL)} rows; * = p or q < .05; '
      f'saved to {SANDBOX / "all_tests_by_cohort.csv"}):')
with pd.option_context('display.max_rows', 500, 'display.max_colwidth', 40):
    display(ALL)

A0 vs V3 Table 1 (cells that differ):
  pt vs pt | between-category: V3 "—"  sandbox "FFA"

── A0_record   (controls 36, LH-intact 12, RH-intact 12)
                  peak selectivity    selective voxels    between-category own side
ctrl L vs R                PPA, LO  VWFA, FFA, PPA, LO      VWFA, FFA, PPA   +0.098
LH-intact vs ctrl               LO                  LO                 FFA   +0.002
RH-intact vs ctrl                —             PPA, LO  VWFA, FFA, PPA, LO   -0.008
pt vs pt                         —                   —                 FFA        —

── A1_no010   (controls 36, LH-intact 12, RH-intact 11)
                  peak selectivity    selective voxels    between-category own side
ctrl L vs R          VWFA, PPA, LO  VWFA, FFA, PPA, LO      VWFA, FFA, PPA   +0.098
LH-intact vs ctrl               LO                  LO                 FFA   +0.002
RH-intact vs ctrl                —             PPA, LO  VWFA, FFA, PPA, LO   -0.021
pt vs pt                         —    

,measure,comparison,test,A0_record,A1_no010,A2_no010_hemi,A3_prisma
0,peak_selectivity,ctrl L vs R,omnibus,p=0.011 *,p=0.011 *,p=0.011 *,p=0.166
1,peak_selectivity,ctrl L vs R,mean over ROIs,d=-0.13 p=0.433,d=-0.13 p=0.427,d=-0.13 p=0.433,d=-0.40 p=0.172
2,peak_selectivity,ctrl L vs R,VWFA,d=+0.36 q=0.050,d=+0.36 q=0.049 *,d=+0.36 q=0.053,d=+0.04 q=0.904
3,peak_selectivity,ctrl L vs R,FFA,d=-0.30 q=0.075,d=-0.30 q=0.081,d=-0.30 q=0.077,d=-0.48 q=0.217
4,peak_selectivity,ctrl L vs R,PPA,d=-0.42 q=0.032 *,d=-0.42 q=0.035 *,d=-0.42 q=0.035 *,d=-0.40 q=0.240
5,peak_selectivity,ctrl L vs R,LO,d=+0.42 q=0.032 *,d=+0.42 q=0.035 *,d=+0.42 q=0.035 *,d=+0.49 q=0.217
6,peak_selectivity,LH-intact vs ctrl,omnibus,p=0.104,p=0.104,p=0.100,p=0.074
7,peak_selectivity,LH-intact vs ctrl,mean over ROIs,d=-0.71 p=0.040 *,d=-0.71 p=0.041 *,d=-1.30 p=0.006 *,d=-0.96 p=0.048 *
8,peak_selectivity,LH-intact vs ctrl,VWFA,d=-0.20 q=0.546,d=-0.20 q=0.544,d=-0.75 q=0.130,d=-0.13 q=0.791
9,peak_selectivity,LH-intact vs ctrl,FFA,d=-0.38 q=0.354,d=-0.38 q=0.356,d=-0.64 q=0.153,d=-0.67 q=0.198


## 7 · Guard

In [7]:
# ─── Guard: the manuscript files are unchanged ───────────────────────────────
HASH_AFTER = {str(p): md5(p) for p in GUARDED}
_changed = [p for p in HASH_BEFORE if HASH_BEFORE[p] != HASH_AFTER[p]]
print('manuscript inputs and outputs unchanged' if not _changed
      else 'CHANGED DURING THIS RUN: ' + ', '.join(_changed))

# any other file under the repo written since the config cell ran
_new = sorted(str(p.relative_to(REPO)) for p in REPO.rglob('*')
              if p.is_file() and p.stat().st_mtime > T_START
              and SANDBOX not in p.parents and '.git' not in p.parts
              and '.ipynb_checkpoints' not in p.parts)
print('files written outside sandbox/: ' + (', '.join(_new) if _new else 'none'))

manuscript inputs and outputs unchanged
files written outside sandbox/: none
